In [4]:
import torch
from torch.utils.data import Dataset, DataLoader
import sentencepiece as spm
import csv
from torch.nn.utils.rnn import pad_sequence

In [5]:
PAD =0
UNK =1
BOS =2
EOS =3

class Seq2SeqDataset(Dataset):
    def __init__(self, tsv_path, sp_model_path):
        self.sp = spm.SentencePieceProcessor(model_file=sp_model_path)
        self.pairs = []
        with open(tsv_path, "r", encoding="utf-8") as f:
            reader = csv.reader(f, delimiter="\t", quoting=csv.QUOTE_NONE, escapechar="\\")
            for row in reader:
                if len(row) == 2:
                    self.pairs.append((row[0], row[1]))



    def __len__(self):
        return len(self.pairs)


    

    def __getitem__(self, idx):
        src_text, tgt_text = self.pairs[idx]
        src_ids = self.sp.encode(src_text, out_type=int)               
        tgt_ids = [BOS] + self.sp.encode(tgt_text, out_type=int) + [EOS]
        return torch.tensor(src_ids, dtype=torch.long), torch.tensor(tgt_ids, dtype=torch.long)

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")

drive_dir = "/content/drive/MyDrive/GenAI-Dataset"
train_path = os.path.join(drive_dir, "train.tsv")
valid_path = os.path.join(drive_dir, "valid.tsv")
wiki_test_path = os.path.join(drive_dir, "wiki_test.tsv")

Mounted at /content/drive


In [ ]:
def collate_fn(batch):
    src_seqs, tgt_seqs = zip(*batch)

    src_lengths = torch.tensor([len(s) for s in src_seqs], dtype=torch.long)
    tgt_lengths = torch.tensor([len(t) for t in tgt_seqs], dtype=torch.long)

    src_padded = pad_sequence(src_seqs, batch_first=True, padding_value=PAD)
    tgt_padded = pad_sequence(tgt_seqs, batch_first=True, padding_value=PAD)

    return src_padded, src_lengths, tgt_padded, tgt_lengths


sp_model_path = "/content/drive/MyDrive/GenAI-Dataset/ur_sp.model"
train_ds = Seq2SeqDataset(train_path, sp_model_path)
valid_ds = Seq2SeqDataset(valid_path, sp_model_path)
wiki_test_ds = Seq2SeqDataset(wiki_test_path, sp_model_path)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_ds, batch_size=64, shuffle=False, collate_fn=collate_fn)
wiki_test_loader = DataLoader(wiki_test_ds, batch_size=64, shuffle=False, collate_fn=collate_fn)

batch = next(iter(train_loader))
src, src_len, tgt, tgt_len = batch
print(src.shape, src_len.shape, tgt.shape, tgt_len.shape)
print(src[0])
print(tgt[0])   

FileNotFoundError: [Errno 2] No such file or directory: 'train.tsv'